# QLOP — Rebuild Model4 (Course Recommendation Two-Tower)

**Run this notebook on Kaggle to rebuild `model4_savedmodel` with proper `saved_model.pb`.**

**Upload dataset dulu:**
1. Di Kaggle, klik `+ Add Data` → `Upload` → upload file `synthetic_demand_course_model4.npz`
   (lokasi: `ai_engine/assets/recommendation/data/synthetic_demand_course_model4.npz`)
2. Setelah selesai, download output zip dan ekstrak ke `ai_engine/assets/recommendation/tf_models/model4_savedmodel/`

In [ ]:
import os, zipfile
import numpy as np
import tensorflow as tf

print('TF version:', tf.__version__)

# ── Sesuaikan path jika nama dataset berbeda di Kaggle ──
NPZ_PATH = '/kaggle/input/qlop-model4-data/synthetic_demand_course_model4.npz'

if not os.path.exists(NPZ_PATH):
    # fallback: cari di semua input dirs
    import glob
    found = glob.glob('/kaggle/input/**/*.npz', recursive=True)
    print('NPZ files found:', found)
    NPZ_PATH = found[0] if found else NPZ_PATH

npz = np.load(NPZ_PATH)
print('Keys:', list(npz.keys()))

demand_vectors  = npz['demand_vectors']    # (15000, 489)
course_vecs_all = npz['course_vectors']    # (1980, 1999)
course_indices  = npz['course_indices']    # (15000,)
targets         = npz['targets'].reshape(-1, 1)  # (15000, 1)

N_DEMAND = demand_vectors.shape[1]   # 489
N_COURSE = course_vecs_all.shape[1]  # 1999
print(f'N_DEMAND={N_DEMAND}, N_COURSE={N_COURSE}, samples={len(demand_vectors)}')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Model4 Architecture  (deduced from checkpoint variable shapes)
#   DemandTower : 489 → 256 → 128 → 64 → 32
#   CourseTower : 1999 → 256 → 128 → 64 → 32
#   Interaction : concat([d, c, d*c, d-c]) → 128 → Dense(32) → bilinear W[32,32] → sigmoid
# ─────────────────────────────────────────────────────────────

EMBED_DIM = 32

class FourInteractionModel(tf.keras.Model):
    def __init__(self, n_demand, n_course, embed_dim=EMBED_DIM):
        super().__init__(name='Model4_CourseRec')

        def make_tower(input_dim, name):
            return tf.keras.Sequential([
                tf.keras.layers.Dense(256, activation='relu', input_shape=(input_dim,)),
                tf.keras.layers.Dense(128, activation='relu'),
                tf.keras.layers.Dense(64,  activation='relu'),
                tf.keras.layers.Dense(embed_dim, activation='relu'),
            ], name=name)

        self.demand_tower = make_tower(n_demand, 'DemandTower')
        self.course_tower = make_tower(n_course, 'CourseTower')

        # Interaction: concat([d, c, d*c, d-c]) = 4 * embed_dim = 128
        self.interaction_dense = tf.keras.layers.Dense(embed_dim, name='interaction_dense')
        self.W = self.add_weight(
            shape=(embed_dim, embed_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='bilinear_W'
        )
        self.output_act = tf.keras.layers.Activation('sigmoid', dtype=tf.float32, name='output_sigmoid')

    def call(self, inputs, training=False):
        demand_vec, course_vec = inputs
        demand_vec = tf.cast(demand_vec, tf.float32)
        course_vec = tf.cast(course_vec, tf.float32)

        d = self.demand_tower(demand_vec, training=training)
        c = self.course_tower(course_vec, training=training)

        interact = tf.concat([d, c, d * c, d - c], axis=-1)      # (batch, 128)
        hidden   = self.interaction_dense(interact)               # (batch, 32)
        score    = tf.reduce_sum(hidden * (c @ self.W), axis=-1, keepdims=True)  # (batch, 1)
        return self.output_act(score)


model4 = FourInteractionModel(N_DEMAND, N_COURSE)
# warm-up
_ = model4([np.zeros((1, N_DEMAND), np.float32), np.zeros((1, N_COURSE), np.float32)])
model4.summary()

In [ ]:
import datetime

# Training data: pair each demand vector with its matched course vector
course_vecs_selected = course_vecs_all[course_indices]   # (15000, 1999)

# Train / val split (80/20)
N = len(demand_vectors)
idx = np.random.permutation(N)
split = int(N * 0.8)
train_idx, val_idx = idx[:split], idx[split:]

X_train = [demand_vectors[train_idx], course_vecs_selected[train_idx]]
y_train = targets[train_idx]
X_val   = [demand_vectors[val_idx],   course_vecs_selected[val_idx]]
y_val   = targets[val_idx]

model4.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='mse',
    metrics=['mae']
)

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
]

history = model4.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=256,
    callbacks=callbacks,
    verbose=1,
)
print('Training complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Export sebagai TF SavedModel dengan serving signature
# yang dipakai model_loader.py:
#   infer4(args_0=demand_f16, args_0_1=course_f16)
#   output key: list(out4.values())[0]
# ─────────────────────────────────────────────────────────────

EXPORT_PATH = '/kaggle/working/model4_savedmodel'

@tf.function(input_signature=[
    tf.TensorSpec([None, N_DEMAND], tf.float16, name='args_0'),
    tf.TensorSpec([None, N_COURSE], tf.float16, name='args_0_1'),
])
def serving_fn(args_0, args_0_1):
    result = model4([args_0, args_0_1], training=False)
    return {'output_0': result}

tf.saved_model.save(
    model4,
    EXPORT_PATH,
    signatures={'serving_default': serving_fn},
)
print(f'Model saved to {EXPORT_PATH}')

# List exported files
for root, dirs, files in os.walk(EXPORT_PATH):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ── Verifikasi serving signature ──
loaded = tf.saved_model.load(EXPORT_PATH)
infer = loaded.signatures['serving_default']

test_out = infer(
    args_0   = tf.constant(np.zeros((5, N_DEMAND), np.float16)),
    args_0_1 = tf.constant(np.zeros((5, N_COURSE), np.float16)),
)
scores = list(test_out.values())[0].numpy().flatten()
print(f'Serving OK — output shape {scores.shape}, sample scores: {scores[:5]}')

In [ ]:
# ── Zip untuk didownload ──
ZIP_PATH = '/kaggle/working/model4_savedmodel.zip'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(EXPORT_PATH):
        for f in files:
            full = os.path.join(root, f)
            arcname = os.path.relpath(full, EXPORT_PATH)
            zf.write(full, arcname)

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f'Zipped to {ZIP_PATH} ({size_mb:.1f} MB)')
print('Download dari Kaggle Output panel, lalu ekstrak ke:')
print('  ai_engine/assets/recommendation/tf_models/model4_savedmodel/')